Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print(" Libraries loaded successfully")

 Libraries loaded successfully


 Load Dataset

In [ ]:
df = pd.read_excel("/content/Final data  (3).xlsx")
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df['Departure_DateTime'].min().date()} to {df['Departure_DateTime'].max().date()}")
print(f"\nAirlines: {df['Airline_Name'].value_counts().to_dict()}")
df.head(3)

Dataset shape: 40000 rows × 41 columns
Date range: 2024-01-01 to 2025-12-01

Airlines: {'Saudia': 18016, 'Flynas': 14054, 'Flyadeal': 7930}


,Flight_ID,Airline_Name,Departure_DateTime,Arrival_DateTime,Issue_Flag,Destination_City,Origin_City,Tourist_Flow,Delay_Category,Satisfaction_Index,...,Flight_Frequency_Per_Week,Social_Media_Engagement,Event_Impact_Flag,Airport_Crowd_Level,Ticket_Type,Customer_Segment,Complaints_Logged,Delay_Reason,Connection_Flag,Loyalty_Status
0,FLT1001,Saudia,2024-08-29 08:36:00,2024-08-29 15:13:00,0,RUH,JED,3978,On-time,86,...,54,3,0,Medium,Non-refundable,Other,0,No Delay,Connecting,Not Enrolled
1,FLT1002,Saudia,2025-06-17 16:54:00,2025-06-17 22:17:00,0,JED,RUH,1362,Long Delay,50,...,9,1,0,Low,Refundable,Leisure,0,No Delay,Connecting,Gold
2,FLT1009,Flyadeal,2024-05-12 09:24:00,2024-05-12 15:57:00,0,GIZ,RUH,2227,On-time,80,...,54,5,0,Medium,Refundable,Leisure,0,No Delay,Connecting,Gold


 Data Quality Audit



In [ ]:

# 1. صححنا  القيم الفارغه
print("\n Missing Values:")
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    for col, count in missing_cols.items():
        print(f"   {col}: {count} ({count/len(df)*100:.1f}%)")
else:
    print("   No missing values found (except Event_Season which is expected)")
print(f"   Event_Season nulls: {df['Event_Season'].isnull().sum()} ({df['Event_Season'].isnull().sum()/len(df)*100:.1f}%)")

# 2. Duplicates
print(f"\n Duplicate Flight_IDs: {df['Flight_ID'].duplicated().sum()} out of {len(df)}")

# 3. Data types
print(f"\n Date columns dtype check:")
print(f"   Departure_DateTime: {df['Departure_DateTime'].dtype}")
print(f"   Arrival_DateTime: {df['Arrival_DateTime'].dtype}")


 Missing Values:
   Event_Season: 10706 (26.8%)
   Event_Season nulls: 10706 (26.8%)

 Duplicate Flight_IDs: 5046 out of 40000

 Date columns dtype check:
   Departure_DateTime: datetime64[ns]
   Arrival_DateTime: datetime64[ns]


 Inconsistency Detection

In [ ]:
print(" INCONSISTENCY #1: Delay_Category vs Delay_Minutes")

for cat in df['Delay_Category'].unique():
    subset = df[df['Delay_Category'] == cat]
    print(f"  {cat:15s} → Delay: min={subset['Delay_Minutes'].min()}, max={subset['Delay_Minutes'].max()}, mean={subset['Delay_Minutes'].mean():.1f}")

ontime_with_delay = df[(df['Delay_Category'] == 'On-time') & (df['Delay_Minutes'] > 0)].shape[0]
print(f"\n   'On-time' category but Delay_Minutes > 0: {ontime_with_delay} records")
print(f"   VERDICT: Delay_Category is BROKEN — no correlation with actual delay")

 INCONSISTENCY #1: Delay_Category vs Delay_Minutes
  On-time         → Delay: min=0, max=180, mean=12.3
  Long Delay      → Delay: min=0, max=180, mean=12.7
  Short Delay     → Delay: min=0, max=180, mean=14.2

   'On-time' category but Delay_Minutes > 0: 4808 records
   VERDICT: Delay_Category is BROKEN — no correlation with actual delay


In [ ]:
print(" INCONSISTENCY #2: Season32 vs actual dates")
print("-" * 50)
print(f"  Season3 values: {df['Season3'].unique().tolist()}")
print(f"  Season32 values: {df['Season32'].unique().tolist()}")

# Verify Season3 against actual months
def get_correct_season(dt):
    month = dt.month
    if month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    elif month in [9, 10, 11]: return 'Autumn'
    else: return 'Winter'

correct_season = df['Departure_DateTime'].apply(get_correct_season)
s3_match = (df['Season3'] == correct_season).sum()
s32_match = (df['Season32'] == correct_season).sum()

print(f"\n  Season3 matches actual month-season: {s3_match}/{len(df)} ({s3_match/len(df)*100:.0f}%) ")
print(f"  Season32 matches actual month-season: {s32_match}/{len(df)} ({s32_match/len(df)*100:.0f}%) ")
print(f"\n   VERDICT: Season32 is BROKEN — random assignment, will be dropped")

 INCONSISTENCY #2: Season32 vs actual dates
--------------------------------------------------
  Season3 values: ['Summer', 'Spring', 'Autumn', 'Winter']
  Season32 values: ['Winter', 'Summer', 'Umrah', 'Riyadh Season', 'Hajj']

  Season3 matches actual month-season: 40000/40000 (100%) 
  Season32 matches actual month-season: 6324/40000 (16%) 

   VERDICT: Season32 is BROKEN — random assignment, will be dropped


In [ ]:
print(" INCONSISTENCY #3: Satisfaction_Index vs Flight_Status")
print("-" * 50)
for status in ['On-time', 'Delayed', 'Cancelled']:
    s = df[df['Flight_Status'] == status]['Satisfaction_Index']
    print(f"  {status:12s} → mean={s.mean():.1f}, median={s.median()}")

print(f"\n   VERDICT: Satisfaction_Index has ZERO correlation with flight status")
print(f"     On-time (70.1) ≈ Delayed (70.2) ≈ Cancelled (70.2) — makes no sense")

 INCONSISTENCY #3: Satisfaction_Index vs Flight_Status
--------------------------------------------------
  On-time      → mean=70.1, median=80.0
  Delayed      → mean=70.2, median=80.0
  Cancelled    → mean=70.2, median=80.0

   VERDICT: Satisfaction_Index has ZERO correlation with flight status
     On-time (70.1) ≈ Delayed (70.2) ≈ Cancelled (70.2) — makes no sense


In [ ]:
print(" INCONSISTENCY #4: Operational_Efficiency_Score vs Flight_Status")
print("-" * 50)
for status in ['On-time', 'Delayed', 'Cancelled']:
    s = df[df['Flight_Status'] == status]['Operational_Efficiency_Score']
    print(f"  {status:12s} → mean={s.mean():.1f}, median={s.median()}")

print(f"\n   VERDICT: Efficiency score has ZERO correlation with operations")

 INCONSISTENCY #4: Operational_Efficiency_Score vs Flight_Status
--------------------------------------------------
  On-time      → mean=75.5, median=88.2
  Delayed      → mean=75.5, median=88.6
  Cancelled    → mean=75.4, median=88.4

   VERDICT: Efficiency score has ZERO correlation with operations


In [ ]:
print(" INCONSISTENCY #5: Event_Season vs Season32")
print("-" * 50)
for event in df['Event_Season'].dropna().unique():
    subset = df[df['Event_Season'] == event]
    print(f"  {event} → Season32: {subset['Season32'].value_counts().head(3).to_dict()}")

print(f"\n   VERDICT: Event_Season is randomly distributed across Season32")

print(f"\n INCONSISTENCY #6: Cancelled flights in Delay_Category")
print("-" * 50)
cancelled = df[df['Flight_Status'] == 'Cancelled']
print(f"  Cancelled flight Delay_Category: {cancelled['Delay_Category'].value_counts().to_dict()}")
print(f"   Cancelled flights labeled as 'On-time' ({(cancelled['Delay_Category']=='On-time').sum()}) and 'Long Delay' ({(cancelled['Delay_Category']=='Long Delay').sum()})")

 INCONSISTENCY #5: Event_Season vs Season32
--------------------------------------------------
  Hajj → Season32: {'Summer': 1213, 'Winter': 1085, 'Umrah': 695}
  Umrah → Season32: {'Summer': 2522, 'Winter': 2014, 'Umrah': 1341}
  Riyadh_Season → Season32: {'Summer': 6642, 'Winter': 5714, 'Umrah': 3732}

   VERDICT: Event_Season is randomly distributed across Season32

 INCONSISTENCY #6: Cancelled flights in Delay_Category
--------------------------------------------------
  Cancelled flight Delay_Category: {'On-time': 2635, 'Long Delay': 1680, 'Short Delay': 92}
   Cancelled flights labeled as 'On-time' (2635) and 'Long Delay' (1680)


Modification of Delay Category


In [ ]:
print("Before:", df['Delay_Category'].value_counts().to_dict())

def fix_delay_category(row):
    if row['Flight_Status'] == 'Cancelled':
        return 'Cancelled'
    elif row['Delay_Minutes'] == 0:
        return 'On-time'
    elif row['Delay_Minutes'] <= 30:
        return 'Short Delay'
    else:
        return 'Long Delay'

df['Delay_Category'] = df.apply(fix_delay_category, axis=1)

print("After: ", df['Delay_Category'].value_counts().to_dict())

# Verify
assert (df[df['Flight_Status'] == 'On-time']['Delay_Minutes'] == 0).all()
assert (df[df['Flight_Status'] == 'Cancelled']['Delay_Category'] == 'Cancelled').all()
print("\n Delay_Category is now consistent with Delay_Minutes and Flight_Status")

Before: {'On-time': 24102, 'Long Delay': 15099, 'Short Delay': 799}
After:  {'On-time': 27636, 'Long Delay': 4935, 'Cancelled': 4407, 'Short Delay': 3022}

 Delay_Category is now consistent with Delay_Minutes and Flight_Status


In [ ]:
def assign_event_season(dt):
    month = dt.month
    year = dt.year

    # Hajj periods
    if year == 2024 and month == 6:
        return 'Hajj'
    elif year == 2025 and month == 6:
        return 'Hajj'

    # Umrah peak (Ramadan)
    elif year == 2024 and month in [3, 4]:
        return 'Umrah'
    elif year == 2025 and month in [2, 3]:
        return 'Umrah'

    # Riyadh Season (Oct to Mar)
    elif month in [10, 11, 12, 1, 2]:
        return 'Riyadh_Season'

    # Summer holidays
    elif month in [7, 8]:
        return 'Summer_Holiday'

    else:
        return None

df['Event_Season'] = df['Departure_DateTime'].apply(assign_event_season)

print("New Event_Season distribution:")
print(df['Event_Season'].value_counts())
print(f"\nNo event period: {df['Event_Season'].isnull().sum()} records")
print("\n Event_Season reassigned based on real dates")

New Event_Season distribution:
Event_Season
Riyadh_Season     13888
Summer_Holiday     7048
Umrah              6762
Hajj               3486
Name: count, dtype: int64

No event period: 8816 records

 Event_Season reassigned based on real dates


Recalculate Satisfaction

In [ ]:
def recalculate_satisfaction(row):
    base = 50

    # Flight status impact
    if row['Flight_Status'] == 'Cancelled':
        base -= 30
    elif row['Flight_Status'] == 'Delayed':
        delay_penalty = min(row['Delay_Minutes'] / 6, 25)
        base -= delay_penalty
    else:
        base += 15

    # Onboard service (1-5 → -10 to +15)
    base += (row['Onboard_Service_Rating'] - 3) * 5 + 5

    # Wait times
    avg_wait = (row['Checkin_Time_Minutes'] + row['Boarding_Time_Minutes'] + row['Baggage_Claim_Time_Minutes']) / 3
    if avg_wait < 20:
        base += 8
    elif avg_wait < 30:
        base += 3
    elif avg_wait > 40:
        base -= 8

    # Complaints & issues
    base -= row['Complaints_Logged'] * 3
    if row['Issue_Flag'] == 1:
        base -= 5

    # Weather
    if row['Weather_Condition2'] in ['Sandstorm', 'Fog', 'Rain']:
        base -= 2

    noise = np.random.normal(0, 3)
    return int(np.clip(base + noise, 0, 100))

np.random.seed(42)
df['Satisfaction_Index'] = df.apply(recalculate_satisfaction, axis=1)

print("New Satisfaction by Flight_Status:")
for status in ['On-time', 'Delayed', 'Cancelled']:
    s = df[df['Flight_Status'] == status]['Satisfaction_Index']
    print(f"  {status:12s} → mean={s.mean():.1f}, median={s.median()}, std={s.std():.1f}")

print("\n Satisfaction_Index now correlates with operational performance")

New Satisfaction by Flight_Status:
  On-time      → mean=78.7, median=79.0, std=5.9
  Delayed      → mean=41.9, median=48.0, std=18.5
  Cancelled    → mean=1.6, median=0.0, std=3.2

 Satisfaction_Index now correlates with operational performance


Recalculate Efficiency

In [ ]:
def recalculate_efficiency(row):
    score = 85

    if row['Flight_Status'] == 'Cancelled':
        score -= 40
    elif row['Flight_Status'] == 'Delayed':
        score -= min(row['Delay_Minutes'] / 4, 30)

    total_wait = row['Checkin_Time_Minutes'] + row['Boarding_Time_Minutes'] + row['Baggage_Claim_Time_Minutes']
    if total_wait > 100:
        score -= 10
    elif total_wait > 80:
        score -= 5

    if row['Issue_Flag'] == 1:
        score -= 8
    score -= row['Complaints_Logged'] * 2

    if row['Load_Factor'] > 90:
        score -= 3

    noise = np.random.normal(0, 2)
    return round(np.clip(score + noise, 0, 100), 1)

np.random.seed(42)
df['Operational_Efficiency_Score'] = df.apply(recalculate_efficiency, axis=1)

print("New Efficiency by Flight_Status:")
for status in ['On-time', 'Delayed', 'Cancelled']:
    s = df[df['Flight_Status'] == status]['Operational_Efficiency_Score']
    print(f"  {status:12s} → mean={s.mean():.1f}, std={s.std():.1f}")

print("\n Operational_Efficiency_Score now correlates with operations")

New Efficiency by Flight_Status:
  On-time      → mean=79.7, std=6.2
  Delayed      → mean=62.0, std=15.0
  Cancelled    → mean=24.9, std=7.9

 Operational_Efficiency_Score now correlates with operations


Fixing Flight Duplication Issue



In [ ]:
print(f"Before: {df['Flight_ID'].nunique()} unique IDs out of {len(df)} rows")

df['Flight_ID'] = [f'FLT{i:06d}' for i in range(1, len(df) + 1)]

print(f"After:  {df['Flight_ID'].nunique()} unique IDs out of {len(df)} rows")
assert df['Flight_ID'].nunique() == len(df)
print("\n All Flight_IDs are now unique")

Before: 34954 unique IDs out of 40000 rows
After:  40000 unique IDs out of 40000 rows

 All Flight_IDs are now unique


 Event Impact Flag

In [ ]:
df['Event_Impact_Flag'] = df['Event_Season'].notna().astype(int)
print(f"Event_Impact_Flag distribution: {df['Event_Impact_Flag'].value_counts().to_dict()}")
print("\n Event_Impact_Flag aligned with Event_Season")

Event_Impact_Flag distribution: {1: 31184, 0: 8816}

 Event_Impact_Flag aligned with Event_Season


 Feature Engineering


In [ ]:
# 1. Flight Duration
df['Flight_Duration_Hours'] = (df['Arrival_DateTime'] - df['Departure_DateTime']).dt.total_seconds() / 3600
df['Flight_Duration_Hours'] = df['Flight_Duration_Hours'].clip(lower=0.5, upper=12)
print(f" Flight_Duration_Hours: mean={df['Flight_Duration_Hours'].mean():.1f}h, min={df['Flight_Duration_Hours'].min():.1f}h, max={df['Flight_Duration_Hours'].max():.1f}h")

# 2. Time-based features
df['Departure_Hour'] = df['Departure_DateTime'].dt.hour
df['Departure_Month'] = df['Departure_DateTime'].dt.month
df['Departure_Year'] = df['Departure_DateTime'].dt.year
df['Departure_Quarter'] = df['Departure_DateTime'].dt.quarter
print(" Extracted: Departure_Hour, Departure_Month, Departure_Year, Departure_Quarter")

# 3. Time Period
def get_time_period(hour):
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'

df['Time_Period'] = df['Departure_Hour'].apply(get_time_period)
print(f" Time_Period: {df['Time_Period'].value_counts().to_dict()}")

# 4. Route
df['Route'] = df['Origin_City'] + ' → ' + df['Destination_City']
print(f" Route: {df['Route'].nunique()} unique routes")

# 5. Total Wait Time
df['Total_Wait_Minutes'] = df['Checkin_Time_Minutes'] + df['Boarding_Time_Minutes'] + df['Baggage_Claim_Time_Minutes']
print(f" Total_Wait_Minutes: mean={df['Total_Wait_Minutes'].mean():.1f} min")

# 6. Delay Risk Binary (classification target)
df['Delay_Risk'] = (df['Flight_Status'] == 'Delayed').astype(int)
print(f" Delay_Risk: {df['Delay_Risk'].value_counts().to_dict()}")

# 7. Satisfaction Level
def sat_level(score):
    if score >= 75: return 'High'
    elif score >= 50: return 'Medium'
    else: return 'Low'

df['Satisfaction_Level'] = df['Satisfaction_Index'].apply(sat_level)
print(f" Satisfaction_Level: {df['Satisfaction_Level'].value_counts().to_dict()}")

# 8. Is Peak Season
df['Is_Peak_Season'] = df['Event_Season'].isin(['Hajj', 'Umrah', 'Riyadh_Season']).astype(int)
print(f" Is_Peak_Season: {df['Is_Peak_Season'].value_counts().to_dict()}")

# 9. Is Weekend (Thursday/Friday in Saudi Arabia)
df['Is_Weekend'] = df['Day_of_Week'].isin(['Thursday', 'Friday']).astype(int)
print(f" Is_Weekend: {df['Is_Weekend'].value_counts().to_dict()}")

# 10. Weather Severity
def weather_severity(condition):
    if condition in ['Sandstorm', 'Fog']: return 'Severe'
    elif condition in ['Rain', 'Cloudy', 'Humid']: return 'Moderate'
    else: return 'Clear'

df['Weather_Severity'] = df['Weather_Condition2'].apply(weather_severity)
print(f" Weather_Severity: {df['Weather_Severity'].value_counts().to_dict()}")

# 11. Service Quality Score (composite)
df['Service_Quality_Score'] = (
    (5 - df['Checkin_Time_Minutes'] / 45 * 5) * 0.2 +
    (5 - df['Boarding_Time_Minutes'] / 50 * 5) * 0.2 +
    (5 - df['Baggage_Claim_Time_Minutes'] / 60 * 5) * 0.2 +
    df['Onboard_Service_Rating'] * 0.4
).round(2)
print(f" Service_Quality_Score: mean={df['Service_Quality_Score'].mean():.2f}")

 Flight_Duration_Hours: mean=5.4h, min=1.0h, max=12.0h
 Extracted: Departure_Hour, Departure_Month, Departure_Year, Departure_Quarter
 Time_Period: {'Night': 13220, 'Morning': 11713, 'Afternoon': 8348, 'Evening': 6719}
 Route: 32 unique routes
 Total_Wait_Minutes: mean=80.5 min
 Delay_Risk: {0: 32043, 1: 7957}
 Satisfaction_Level: {'High': 21400, 'Medium': 9960, 'Low': 8640}
 Is_Peak_Season: {1: 24136, 0: 15864}
 Is_Weekend: {0: 28604, 1: 11396}
 Weather_Severity: {'Clear': 19486, 'Moderate': 14491, 'Severe': 6023}
 Service_Quality_Score: mean=3.01


Clean Arabic Text

In [ ]:
def clean_arabic_text(text):
    if pd.isna(text):
        return text
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove English text
    text = re.sub(r'[a-zA-Z]+', '', text)
    # Normalize Arabic characters
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    # Remove diacritics
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    # Remove repeated characters (الحروف الزايده)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Remove emojis and special characters
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Tweet_Text_Clean'] = df['Tweet_Text'].apply(clean_arabic_text)

# Text features
df['Text_Length'] = df['Tweet_Text_Clean'].str.len()
df['Word_Count'] = df['Tweet_Text_Clean'].str.split().str.len()

print(" Arabic text preprocessing complete")
print(f"   Avg text length: {df['Text_Length'].mean():.0f} characters")
print(f"   Avg word count:  {df['Word_Count'].mean():.1f} words")

# Show samples
print("\n Sample cleaned texts:")
for i in range(3):
    print(f"   {df['Tweet_Text_Clean'].iloc[i][:80]}...")

 Arabic text preprocessing complete
   Avg text length: 88 characters
   Avg word count:  15.4 words

 Sample cleaned texts:
   ماشاء الله وصلنا بالوقت، الطياره نظيفه والمضيفين بشوشين...
   سرعه في الاجراءات ووصلنا قبل الموعد، شغل احترافي...
   الزحمه كانت متوسطه، تاخير بسيط بس مقبول...


 Final Validation



In [ ]:
checks = []

# 1
c = (df[df['Flight_Status'] == 'On-time']['Delay_Minutes'] == 0).all()
checks.append(('On-time flights → 0 delay minutes', c))

# 2
c = (df[df['Flight_Status'] == 'Delayed']['Delay_Minutes'] > 0).all()
checks.append(('Delayed flights → positive delay minutes', c))

# 3
c = (df[df['Flight_Status'] == 'Cancelled']['Delay_Category'] == 'Cancelled').all()
checks.append(('Cancelled flights → Cancelled category', c))

# 4
on_sat = df[df['Flight_Status'] == 'On-time']['Satisfaction_Index'].mean()
del_sat = df[df['Flight_Status'] == 'Delayed']['Satisfaction_Index'].mean()
can_sat = df[df['Flight_Status'] == 'Cancelled']['Satisfaction_Index'].mean()
c = on_sat > del_sat > can_sat
checks.append((f'Satisfaction: On-time({on_sat:.1f}) > Delayed({del_sat:.1f}) > Cancelled({can_sat:.1f})', c))

# 5
on_eff = df[df['Flight_Status'] == 'On-time']['Operational_Efficiency_Score'].mean()
del_eff = df[df['Flight_Status'] == 'Delayed']['Operational_Efficiency_Score'].mean()
c = on_eff > del_eff
checks.append((f'Efficiency: On-time({on_eff:.1f}) > Delayed({del_eff:.1f})', c))

# 6
c = df['Flight_ID'].nunique() == len(df)
checks.append(('All Flight_IDs unique', c))




for name, result in checks:

    print(f"  {status} {name}")

for name, result in checks:
    print(name, "→", result)
print(f"All {sum(1 for _,r in checks if r)}/{len(checks)} checks passed!")

  Cancelled On-time flights → 0 delay minutes
  Cancelled Delayed flights → positive delay minutes
  Cancelled Cancelled flights → Cancelled category
  Cancelled Satisfaction: On-time(78.7) > Delayed(41.9) > Cancelled(1.6)
  Cancelled Efficiency: On-time(79.7) > Delayed(62.0)
  Cancelled All Flight_IDs unique
On-time flights → 0 delay minutes → True
Delayed flights → positive delay minutes → True
Cancelled flights → Cancelled category → True
Satisfaction: On-time(78.7) > Delayed(41.9) > Cancelled(1.6) → True
Efficiency: On-time(79.7) > Delayed(62.0) → True
All Flight_IDs unique → True
All 6/6 checks passed!
